# Step 2b — Real cross-source address pairs (Overture + Foursquare)

Builds positive address pairs for CT: the same place's address as published by Overture (side A, usually Meta/Microsoft text) vs. Foursquare's own record (side B).

**Pipeline**
1. `ct_places` — Overture places in CT (release `2026-09-23.1`)
2. `ct_bridge` — GERS id → Foursquare / AllThePlaces record ids (same release)
3. `fsq_ct` — Foursquare OS Places records around CT (release `dt=2026-09-15`)
4. `ct_positives` — the 3-way join → positive pairs
5. Analysis of the positives + unit stripping (in progress)

⚠️ Cells marked **RUN ONCE** download from S3 / Hugging Face and take minutes. All outputs are saved to `Overture/`; later cells read the local parquet files.

In [ ]:
import os
import duckdb

con = duckdb.connect()                     # in-memory; tables vanish when the kernel restarts
con.sql("INSTALL httpfs; LOAD httpfs;")    # lets DuckDB read s3:// and https:// paths
con.sql("SET s3_region = 'us-west-2';")    # Overture's public bucket region

## 1. Overture places in CT — `ct_places`

- Bounding box around CT for speed (lets DuckDB skip row groups), then an exact CT filter.
- `region` values were `CT` and `ct` → compare with `UPPER()`.
- 393 places with NULL `region` but a CT ZIP (`06…`) are kept; the rest of the NULL-region rows are NY/MA/RI or have no postcode.
- `confidence` (does the place exist?) is kept as a column, not filtered — it says nothing about address quality.

Result: **193,270 places**, all ids unique.

In [ ]:
# RUN ONCE — scans Overture places on S3 (~11 GB total, but only CT row groups/columns are fetched)
con.sql("""
CREATE TABLE ct_places AS
SELECT id, names.primary AS name, UPPER(addresses[1].freeform) AS freeform, UPPER(addresses[1].locality) AS locality, addresses[1].postcode AS postcode, addresses[1].region AS region, confidence, sources, bbox
FROM read_parquet('s3://overturemaps-us-west-2/release/2026-09-23.1/theme=places/type=place/*.zstd.parquet')
WHERE addresses[1].freeform IS NOT NULL
AND
    bbox.xmin BETWEEN -73.73 AND -71.78 AND
    bbox.ymin BETWEEN 40.98 AND 42.06 AND
    -- parentheses matter: AND binds tighter than OR
    (UPPER(addresses[1].region) = 'CT'
    OR (addresses[1].region IS NULL AND addresses[1].postcode LIKE '06%'))
""")

con.sql("""
    COPY ct_places TO
    'Overture/ct_places_2026-09-23.1.parquet' (FORMAT parquet)
""")

In [ ]:
# Check: expect 193,270
con.sql("""
    SELECT COUNT(*) FROM 
    read_parquet('Overture/ct_places_2026-09-23.1.parquet')
""")

## 2. Bridge links — `ct_bridge`

Overture's published record keeps only the provider that "won" the merge (`sources[1]`, verified `property = ''` for every row). The **bridge files** list every provider record that was merged into each GERS id — that's what turns single records into pairs.

- `provider` = whose record this bridge row points to (side B candidate; comes from the `provider=…` folder name)
- `published_from` = whose address Overture published (side A)
- `provider = published_from` → **self-pair** (same text on both sides) → not usable

Result: 58,694 rows (50,181 Foursquare + 8,513 AllThePlaces, one record per provider per place). Self-pairs: 12,483 FSQ + 2,067 ATP → **44,144 usable**.

In [ ]:
# RUN ONCE — scans the FSQ (~800 MB) and ATP (~210 MB) bridge files on S3
# UNION ALL stacks the two results; columns are matched by POSITION, so both SELECTs must list them in the same order
con.sql("""
    CREATE TABLE ct_bridge AS 

    SELECT p.id AS gers_id, b.record_id, b.provider, LOWER(p.sources[1].dataset) AS published_from
    FROM read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p
    JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=foursquare/theme=places/type=place/*.parquet') AS b
    ON p.id = b.id

    UNION ALL

    SELECT p.id AS gers_id, b.record_id, b.provider, LOWER(p.sources[1].dataset) AS published_from
    FROM read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p
    JOIN read_parquet('s3://overturemaps-us-west-2/bridgefiles/2026-09-23.1/provider=alltheplaces/theme=places/type=place/*.parquet') AS b
    ON p.id = b.id;

    COPY ct_bridge TO 'Overture/ct_bridge_2026-09-23.1.parquet' (FORMAT parquet)
""")

In [ ]:
# Check: who supplied side A vs. which provider record we link to; rows where the two match are self-pairs
con.sql("""
SELECT provider, published_from, COUNT(*) as N
FROM read_parquet('Overture/ct_bridge_2026-09-23.1.parquet')
GROUP BY provider, published_from
""")

## 3. Foursquare records around CT — `fsq_ct`

Foursquare OS Places is gated on Hugging Face: the token comes from `HF_TOKEN` (set in the git-ignored `.env`, loaded by direnv). Never print it.

- **Bounding box only, no region filter.** Some Foursquare places located in CT carry wrong states (`KS`, `CA`, …); a region filter would drop them and lose their pairs. CT-ness was already decided on the Overture side — this table is just a lookup, and the join does the exact selection.
- `region` is kept raw (`CT` / `Connecticut` / `ct` …) — normalize it in feature code, not here.

Coverage check: only **31 of 50,181** bridge Foursquare ids are missing from `fsq_ct` (99.94%) — the id formats match.

In [ ]:
# Register the Hugging Face token with DuckDB (temporary: lasts for this connection only)
con.sql(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

In [ ]:
# RUN ONCE — reads Foursquare OS Places from Hugging Face
con.sql("""
    CREATE TABLE fsq_ct AS 
    SELECT 
    fsq_place_id, name,
    UPPER(address) AS address, 
    UPPER(locality) AS locality,
    region, postcode,
    latitude, longitude, bbox,
    date_closed
    FROM 
    read_parquet('hf://datasets/foursquare/fsq-os-places/release/dt=2026-09-15/places/parquet/*.parquet')
    WHERE 
    bbox.xmin BETWEEN -73.73 AND -71.78 AND
    bbox.ymin BETWEEN 40.98 AND 42.06 
"""
)

con.sql("""
    COPY fsq_ct
    TO 'Overture/fsq_ct_2026-09-15.parquet' (FORMAT PARQUET)
""")

In [ ]:
# Coverage check: LEFT JOIN keeps every bridge row, so misses show up as NULLs (expect 31 True)
con.sql("""
    SELECT(q.fsq_place_id IS NULL) as fsq_id_null, COUNT(*) AS N
    FROM (
        SELECT *
        FROM read_parquet('Overture/ct_bridge_2026-09-23.1.parquet') 
        where provider = 'foursquare'
    ) as p
    LEFT JOIN read_parquet('Overture/fsq_ct_2026-09-15.parquet') as q
    ON p.record_id = q.fsq_place_id
    GROUP BY fsq_id_null
""").df()

## 4. Positive pairs — `ct_positives`

Chain of joins through the bridge (the middle table): `places ── gers_id ── bridge ── record_id ── foursquare`.
Foursquare only for now (AllThePlaces deferred — see open questions). Self-pairs excluded.

Result: **37,680 pairs**. `identical_address` flags the 70% (26,286) whose strings are exactly equal — kept, but report results on the non-identical subset.

In [ ]:
con.sql("""
    DROP TABLE IF EXISTS ct_positives;
    CREATE TABLE ct_positives AS 
    SELECT 
    b.gers_id, f.fsq_place_id,            -- ids kept for tracing pairs in error analysis
    p.freeform AS address_a,              -- _a = Overture (published_from provider)
    f.address AS address_b,               -- _b = Foursquare
    p.locality AS locality_a,
    f.locality AS locality_b,
    p.region AS region_a,
    f.region AS region_b,
    p.postcode AS postcode_a,
    f.postcode AS postcode_b,
    p.name AS name_a,                     -- names are NOT features (address-only scope); for reading pairs only
    f.name AS name_b,
    b.published_from,
    1 AS label,
    address_a = address_b AS identical_address
    FROM read_parquet('Overture/ct_bridge_2026-09-23.1.parquet') AS b
    JOIN read_parquet('Overture/ct_places_2026-09-23.1.parquet') AS p 
    ON b.gers_id = p.id
    JOIN read_parquet('Overture/fsq_ct_2026-09-15.parquet') AS f
    on b.record_id = f.fsq_place_id
    WHERE b.provider = 'foursquare' AND b.provider <> b.published_from;

    COPY ct_positives TO 'Overture/ct_positives_2026-09-23.1.parquet'
    (FORMAT parquet);
""")

## 5. What's in the positives

Eyeball sample of the non-identical pairs. Categories seen: unit/suite added on one side, abbreviations, ranges (`16-18`), ZIP+4, junk (`\N\N`), real typos (`DANBURY EOAD`), locality variants (`SPRAGUE`/`BALTIC`), and different house numbers.

In [ ]:
# Random sample of pairs that actually differ (re-run for a new sample; SELECT setseed(x) first to make it repeatable)
con.sql("""
    SELECT address_a, address_b, locality_a, locality_b, postcode_a, postcode_b, name_a, name_b
    FROM read_parquet('Overture/ct_positives_2026-09-23.1.parquet')
    WHERE NOT identical_address
    ORDER BY random()
    LIMIT 20
""").df()

House-number comparison. Result: 1,177 missing a number on one side (ambiguous — keep, flag) and **1,778 with different numbers** (likely label noise — flag, keep out of training positives, sample for the hand-labeled test set). The 544 rows with NULL `address_b` fall in neither count (NULL fails every comparison).

In [ ]:
# Leading house number on each side; regexp_extract returns '' (not NULL) when there's no match
con.sql(r"""
SELECT 
    COUNT(*) 
        AS total,
    COUNT(*) FILTER (WHERE number_a = '' OR number_b = '') 
        AS missing_number,
    COUNT(*) FILTER 
        (WHERE number_a <> '' AND number_b <> '' AND number_a <> number_b)                 
        AS different_number
FROM  
    ( 
    SELECT 
        regexp_extract(address_a, '^\d+') AS number_a,
        regexp_extract(address_b, '^\d+') AS number_b
    FROM 
        read_parquet('Overture/ct_positives_2026-09-23.1.parquet')
)
"""
)

## 6. Unit stripping (in progress)

Needed for the **building-level** match definition (same street address = same location, units ignored).

- ✅ `#` + alphanumeric token at the end: `\s*#\s*[A-Z0-9]+$` (handles `#1C`, `# 1D`, no trailing space)
- ⬜ designator words: `UNIT 2W`, `STE 109`, `SUITE #2C`, `APT 2`, `STORE #240` — require a word **or** a `#`, otherwise the pattern strips the last word of every address
- ⬜ double spaces, punctuation (`RD.`), junk tokens

In [ ]:
# Test the pattern on rows with letter-suffixed '#' units, originals next to stripped values
con.sql(r"""
    SELECT 
        address_a,
        regexp_replace(address_a, '\s*\#\s*[A-Z0-9]+$', '') AS stripped_a,
        address_b,
        regexp_replace(address_b, '\s*\#\s*[A-Z0-9]+$', '') AS stripped_b,
    FROM 
        read_parquet('Overture/ct_positives_2026-09-23.1.parquet')
    WHERE address_b LIKE '%#%' 
    AND regexp_matches(address_b, '#\s*\d+[A-Z]')
    ORDER BY random()
    LIMIT 10
""").df()